# 06 · Fine-tune neural detectors (Keras 3 + KerasHub, TensorFlow backend)
**Detectors:** mDeBERTa-v3-base (`deberta_v3_base_multi`) and XLM-R-base (`xlm_roberta_base_multi`).

**Experiments (from `config.json`):**
- **E1** monolingual training (6 runs/detector/seed) → also yields the 6×6 cross-lingual transfer matrix (RQ1, RQ2)
- **E2** multilingual (`all`) + leave-one-language-out for the 4 Turkic targets (RQ1, RQ2) — `E2/all` is also the main model for robustness (RQ4)
- **E3** leave-one-generator-family-out, 5 folds (RQ3)
- **E6** *(optional)* multilingual + T3/T4 augmentation (robustness defence)

**No recomputation:** each run trains once and scores *every* evaluation text it needs in the same session; finished runs are skipped (registry + prediction file).
**Session-safe:** if Colab disconnects, just re-run — completed runs are skipped.

**Runtime:** **A100 required** (L4 accepted; T4 refused — out of memory). 60 runs for E1–E3 (3 seeds for E1 and E2/all, 1 seed for LOLO and E3); E2/all is the largest (~8.2k training texts), E1 runs are ~6× smaller.
**Warnings you can ignore:** `shuffle=True ... ignored` and `Converting sparse IndexedSlices to a dense Tensor`.

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "06_train_finetuned_detectors"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()

In [ ]:
!pip -q install "keras-hub==0.32.0" "keras==3.15.1"

In [ ]:
os.environ["KERAS_BACKEND"] = CFG["finetune"]["backend"]          # must be set before importing keras
import keras, keras_hub, tensorflow as tf
log = get_logger(NOTEBOOK); env = env_snapshot(NOTEBOOK)
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
gpu = env["gpu"]; log.info(f"GPU={gpu} keras={keras.__version__} keras_hub={keras_hub.__version__} tf={tf.__version__}")
# v1.2: one training setup for ALL runs (batch 32, bf16). A T4 (16 GB) cannot hold mDeBERTa/XLM-R (250k-token
# vocabularies) at batch 32 and ran out of memory between runs, so it is refused instead of silently changing the setup.
if not any(k in gpu for k in ("A100", "H100", "L4")):
    raise RuntimeError(f"GPU '{gpu}' is not supported for 06. Runtime → Change runtime type → A100 GPU (or L4), then re-run.")
POLICY = "mixed_bfloat16"
keras.mixed_precision.set_global_policy(POLICY)
FT = CFG["finetune"]
BATCH = FT["batch_size"]
log.info(f"precision policy={POLICY} batch={BATCH}")

## Run selection (edit to split work across sessions)

In [ ]:
RUN_EXPS = ["E2", "E3", "E1"]          # priority order; add "E6" for the optional augmentation arm
RUN_DETECTORS = list(FT["models"])      # ["mdeberta", "xlmr"]
# v1.2 compute budget: 3 seeds for the central runs (E1 = transfer matrix, E2/all = main multilingual model);
# 1 seed (SEEDS[0]) for leave-one-language-out (E2/no-L) and leave-one-family-out (E3). Their CIs therefore reflect
# document sampling only (report; seed spread is estimated from the 3-seed runs). 96 -> 60 runs, ~half the GPU time.
def seeds_for(exp, spec):
    return SEEDS if exp == "E1" or (exp == "E2" and spec["name"] == "all") else SEEDS[:1]
FULL_EVAL = {("E2", "all"), ("E6", "all-aug")}   # these runs also score all attacked sets (RQ4); others score val/test/external only

ds = load_dataset()
EVAL = load_eval_texts()
CORE_SETS = {"val", "test", "ext_coling_raw", "ext_coling_norm"}

plan = []
for exp in RUN_EXPS:
    for spec in CFG["experiments"][exp]:
        for det in RUN_DETECTORS:
            for seed in seeds_for(exp, spec):
                rid = make_run_id(exp, det, spec["name"], spec.get("heldout_family") or "none", seed)
                plan.append((rid, exp, spec, det, seed))
def done_with_same_setup(rid):
    """Done AND trained with the current batch/precision (runs from another setup are re-trained and overwritten)."""
    if not is_done(rid):
        return False
    r = load_registry().set_index("run_id").loc[rid]
    prm = json.loads(r.params_json) if isinstance(r.params_json, str) else {}
    return prm.get("batch") == BATCH and prm.get("policy") == POLICY
todo = [p for p in plan if not done_with_same_setup(p[0])]
log.info(f"planned runs {len(plan)} | remaining {len(todo)}")

## Training / scoring functions

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def make_tfds(pre, texts, labels=None, shuffle=False, seed=0, bs=BATCH):
    d = tf.data.Dataset.from_tensor_slices((list(texts), np.asarray(labels, "int32")) if labels is not None else list(texts))
    if shuffle:
        d = d.shuffle(len(texts), seed=seed, reshuffle_each_iteration=True)
    d = d.batch(bs)
    d = d.map(pre, num_parallel_calls=AUTOTUNE)              # preprocessor maps (text[, label]) -> (token dict[, label])
    return d.prefetch(AUTOTUNE)

def run_one(rid, exp, spec, det, seed):
    m = FT["models"][det]
    params = {"exp": exp, "spec": spec, "detector": det, "preset": m["preset"], "seed": seed, "seq_len": FT["seq_len"],
              "batch": BATCH, "epochs": FT["epochs"], "lr": FT["lr"], "wd": FT["weight_decay"], "policy": POLICY}
    register_run(rid, "running", exp=exp, detector=det, train=spec["name"], heldout=spec.get("heldout_family") or "none",
                 seed=seed, params_json=json.dumps(params))
    t0 = time.time()
    tr, va = training_frame(ds, spec)
    if spec.get("augment"):
        tr = augment_train(tr, spec["augment"], seed)
    set_seed(seed)
    cls = getattr(keras_hub.models, m["class"])
    pre = cls.preprocessor_cls.from_preset(m["preset"], sequence_length=FT["seq_len"])
    model = cls.from_preset(m["preset"], num_classes=2, preprocessor=None)
    steps = int(np.ceil(len(tr) / BATCH)) * FT["epochs"]; warm = int(FT["warmup_frac"] * steps)
    sched = keras.optimizers.schedules.CosineDecay(0.0, decay_steps=max(1, steps - warm), warmup_target=FT["lr"], warmup_steps=warm)
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=sched, weight_decay=FT["weight_decay"]),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")])
    hist = model.fit(make_tfds(pre, tr.text, tr.label, shuffle=True, seed=seed),
                     validation_data=make_tfds(pre, va.text, va.label), epochs=FT["epochs"], verbose=2,
                     callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=FT["early_stopping_patience"],
                                                              restore_best_weights=True)])
    # ---- score every evaluation text this run needs (single pass) ----
    sets = set(EVAL.eval_set.unique()) if (exp, spec["name"]) in FULL_EVAL else CORE_SETS
    ev = EVAL[EVAL.eval_set.isin(sets)]
    logits = model.predict(make_tfds(pre, ev.text, bs=BATCH * 2), verbose=0).astype("float32")
    prob = tf.nn.softmax(logits, axis=-1).numpy()[:, 1]
    out = pd.DataFrame({"doc_id": ev.doc_id.values, "score": logits[:, 1] - logits[:, 0], "prob": prob})
    save_parquet(out, pred_path(rid))
    if f"{exp}__{spec['name']}" in FT["save_checkpoint_for"] and seed == SEEDS[0]:
        model.save_to_preset(str(P["models"] / rid))
    best = int(np.argmin(hist.history["val_loss"]))
    register_run(rid, "done", seconds=round(time.time() - t0, 1),
                 notes=json.dumps({"n_train": len(tr), "n_val": len(va), "epochs_run": len(hist.history["loss"]),
                                   "best_epoch": best + 1, "best_val_loss": float(hist.history["val_loss"][best]),
                                   "n_scored": len(out)}))
    del model; keras.backend.clear_session(); gc.collect()

## Smoke test (one tiny run, not registered) — run once per new runtime

In [ ]:
SMOKE = True
if SMOKE:
    m = FT["models"][RUN_DETECTORS[0]]
    cls = getattr(keras_hub.models, m["class"])
    pre = cls.preprocessor_cls.from_preset(m["preset"], sequence_length=64)
    mdl = cls.from_preset(m["preset"], num_classes=2, preprocessor=None)
    mdl.compile(optimizer=keras.optimizers.AdamW(2e-5), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    sm = ds[ds.split == "train"].sample(64, random_state=0)
    mdl.fit(make_tfds(pre, sm.text, sm.label, shuffle=True), epochs=1, verbose=2)
    print(mdl.predict(make_tfds(pre, sm.text.head(8)), verbose=0)[:3])
    del mdl; keras.backend.clear_session(); gc.collect()

## Run the plan

In [ ]:
for i, (rid, exp, spec, det, seed) in enumerate(todo, 1):
    log.info(f"=== [{i}/{len(todo)}] {rid}")
    try:
        run_one(rid, exp, spec, det, seed)
    except tf.errors.ResourceExhaustedError as e:
        register_run(rid, "failed", notes="OOM: " + str(e)[:300])
        log.error("GPU out of memory — STOP. Runtime → Restart session, then re-run this notebook (finished runs are skipped).")
        break
    except Exception as e:
        log.exception(f"run failed: {rid}")
        register_run(rid, "failed", notes=str(e)[:500])
        keras.backend.clear_session(); gc.collect()

reg = load_registry(); reg[reg.run_id.isin([p[0] for p in plan])].groupby(["exp", "detector", "status"]).size()